# Proyecto de Minería de Datos — Turismo y Ocupación Hotelera en Risaralda
**Integrantes:** José Luis Brito Henao · Luis Eduardo Trujillo Agudelo

Este cuaderno contiene el desarrollo estructurado del proyecto, desde el cruce de datos crudos hasta la preparación para modelos predictivos. Está dividido en módulos claros para facilitar su lectura y evaluación.

## Módulo 1: Cruce de Datos (EMA - DANE + RNT)
En esta sección se cargan los archivos crudos, se limpian y se cruzan para generar el `dataset_cruzado.xlsx`. Se corrigieron los conteos de habitaciones y establecimientos.

In [ ]:
import pandas as pd
import numpy as np

# 1. Carga de datos crudos (Ajusta las rutas según tu repositorio)
# rnt = pd.read_excel("data/thwd-ivmp.xlsx")
# rnt = rnt.iloc[:,0].str.split(",", expand=True)
# rnt.columns = [
#     "codigo_rnt", "estado_rnt", "razon_social_establecimiento",
#     "departamento", "cod_dpto", "municipio", "cod_mun", "nit",
#     "categoria", "sub_categoria", "habitaciones", "camas",
#     "num_emp1", "ano"
# ]

# ema = pd.read_excel("data/anex-EMA-jul2026 (1).xlsx", sheet_name=None)

# 2. Limpieza de la hoja de ocupación
# ocupacion = ema["4.2 Porc Mens Ocupación.reg"]
# ocupacion_limpia = ocupacion.copy()
# ocupacion_limpia.columns = ocupacion_limpia.iloc[10]
# ocupacion_limpia = ocupacion_limpia.iloc[11:]
# ocupacion_limpia.columns = ocupacion_limpia.columns.astype(str).str.strip()
# ocupacion_limpia = ocupacion_limpia.loc[:, ocupacion_limpia.columns.notna()]
# ocupacion_limpia["Año"] = ocupacion_limpia["Año"].ffill()
# ocupacion_limpia["Año"] = ocupacion_limpia["Año"].astype(str).str.replace("p", "", regex=False)
# ocupacion_limpia["Año"] = pd.to_numeric(ocupacion_limpia["Año"], errors="coerce")

# 3. Transformación a formato largo (melt)
# columnas_regiones = list(ocupacion_limpia.columns[3:15])
# ocupacion_long = pd.melt(
#     ocupacion_limpia,
#     id_vars=["Año", "Meses"],
#     value_vars=columnas_regiones,
#     var_name="region",
#     value_name="ocupacion"
# )

# --- CORRECCIÓN: Ocupación a float ---
# ocupacion_long["ocupacion"] = (
#     ocupacion_long["ocupacion"].astype(str)
#     .str.replace("%", "", regex=False).str.replace(",", ".", regex=False).str.strip()
# )
# ocupacion_long["ocupacion"] = pd.to_numeric(ocupacion_long["ocupacion"], errors="coerce") / 100

# ocupacion_long = ocupacion_long.rename(columns={"Meses": "mes_nombre"})

# 4. Construir fecha
# mes_map = {"Enero":1,"Febrero":2,"Marzo":3,"Abril":4,"Mayo":5,"Junio":6,"Julio":7,"Agosto":8,"Septiembre":9,"Octubre":10,"Noviembre":11,"Diciembre":12}
# ocupacion_long["mes"] = ocupacion_long["mes_nombre"].map(mes_map)
# ocupacion_long["fecha"] = pd.to_datetime(dict(year=ocupacion_long["Año"], month=ocupacion_long["mes"], day=1))

# 5. RNT y Mapeo
# rnt_hoteles = rnt[rnt["categoria"].str.contains("ALOJAMIENTO", case=False, na=False)].copy()
# for col in rnt_hoteles.columns: rnt_hoteles[col] = rnt_hoteles[col].astype(str).str.replace('"', '', regex=False).str.strip()
# mapa_region = {"ANTIOQUIA": "Antioquia", "ATLANTICO": "Caribe", "CESAR": "Caribe", "MAGDALENA": "Caribe", "LA GUAJIRA": "Caribe", "CALDAS": "Eje Cafetero", "QUINDIO": "Eje Cafetero", "RISARALDA": "Eje Cafetero", "VALLE DEL CAUCA": "Pacífico", "CAUCA": "Pacífico", "NARIÑO": "Pacífico", "CHOCO": "Pacífico", "SANTANDER": "Santanderes", "NORTE DE SANTANDER": "Santanderes", "META": "Llanos Orinoquía", "ARAUCA": "Llanos Orinoquía", "CASANARE": "Llanos Orinoquía", "VICHADA": "Llanos Orinoquía", "AMAZONAS": "Amazonía", "CAQUETA": "Amazonía", "PUTUMAYO": "Amazonía", "GUAINIA": "Amazonía", "GUAVIARE": "Amazonía", "VAUPES": "Amazonía", "BOGOTA": "Bogotá D.C.", "CUNDINAMARCA": "Central", "BOYACA": "Central", "HUILA": "Central", "TOLIMA": "Central", "CORDOBA": "Golfo Morrosquillo y Sabana", "SUCRE": "Golfo Morrosquillo y Sabana", "SAN ANDRES Y PROVIDENCIA": "San Andrés y Providencia", "BOLIVAR": "Cartagena"}
# rnt_hoteles["region"] = rnt_hoteles["departamento"].str.upper().map(mapa_region)

# --- CORRECCIÓN: Agrupación correcta (nunique + sum) ---
# rnt_hoteles["habitaciones"] = pd.to_numeric(rnt_hoteles["habitaciones"], errors="coerce")
# rnt_resumen = (
#     rnt_hoteles.groupby("region", as_index=False)
#     .agg(establecimientos=("codigo_rnt", "nunique"), habitaciones=("habitaciones", "sum"))
# )

# 6. Cruce final
# dataset_final = pd.merge(ocupacion_long, rnt_resumen, on="region", how="left", indicator=True)
# dataset_final.to_excel("dataset_cruzado.xlsx", index=False)
print("Módulo 1: Cruce de datos cargado (código documentado para referencia).")

## Módulo 2: Análisis Exploratorio de Datos (EDA)
En esta sección se cargan los datos ya cruzados y se preparan las variables base para los módulos siguientes.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Cargar el dataset final
df = pd.read_excel("dataset_cruzado.xlsx")
print("Dimensiones del dataset:", df.shape)
display(df.head())

In [ ]:
# Validación rápida de datos
print("=== Tipos de dato ===")
print(df.dtypes)
print("\n=== Valores nulos ===")
print(df.isna().sum())
print("\n=== Ocupación promedio por región ===")
print(df.groupby("region")["ocupacion"].mean().sort_values(ascending=False))

## Módulo 3: Ingeniería de Características (Feature Engineering)
Se crean variables derivadas (trimestre, pandemia, terremoto, categoría de ocupación) y se aplica un pipeline reproducible.

In [ ]:
def preparar_features(data):
    """Pipeline reproducible: limpieza + derivadas + one-hot + escalado."""
    d = data.copy()
    
    # Asegurar tipos numéricos y fecha
    d["ocupacion"] = pd.to_numeric(d["ocupacion"], errors="coerce")
    d["fecha"] = pd.to_datetime(d["fecha"], errors="coerce")
    
    # Variables derivadas
    d["trimestre"] = d["fecha"].dt.quarter
    d["franja_anio"] = np.select(
        [d["Año"] == 2019, d["Año"].isin([2020, 2021])],
        ["pre-pandemia", "pandemia"],
        default="post-pandemia"
    )
    d["es_terremoto"] = (d["fecha"] >= "2026-08-01").astype(int)
    d["ocupacion_categoria"] = pd.cut(
        d["ocupacion"], bins=[-0.01, 0.30, 0.50, 1.01],
        labels=["bajo", "medio", "alto"]
    )
    
    # One-hot encoding
    d = pd.get_dummies(d, columns=["region", "franja_anio", "ocupacion_categoria"], prefix=["region", "franja", "ocup_cat"])
    
    # Escalado min-max de la ocupación
    mn, mx = d["ocupacion"].min(), d["ocupacion"].max()
    d["ocupacion_escalada"] = (d["ocupacion"] - mn) / (mx - mn) if mx > mn else 0.0
    
    return d

df_features = preparar_features(df)
print("Pipeline aplicado con éxito. Dimensiones finales:", df_features.shape)
display(df_features.head())

## Módulo 4: Modelo Predictivo y Evaluación
Se entrena un Árbol de Decisión para predecir la categoría de ocupación (bajo/medio/alto) y se evalúa su rendimiento.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

# Definir variables X e y (excluyendo las que causarían fuga de datos)
cols_excluir = ["ocupacion", "ocupacion_escalada", "ocupacion_categoria", "mes_nombre", "fecha", "region", "franja_anio", "Año", "Mes", "Meses"]
features = [c for c in df_features.columns if c not in cols_excluir and df_features[c].dtype != "object"]

X = df_features[features].fillna(0)
y = df_features["ocupacion_categoria"].astype(str)

# Separar datos
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Entrenar modelo
modelo = DecisionTreeClassifier(max_depth=4, random_state=42)
modelo.fit(X_train, y_train)

# Predecir y evaluar
pred_test = modelo.predict(X_test)
print("Accuracy en Test:", round(accuracy_score(y_test, pred_test), 3))
print("\nReporte de Clasificación:")
print(classification_report(y_test, pred_test, zero_division=0))